# Evaluating RAG Pipeline Output Quality

This notebook is the runnable companion to [Evaluating Pipeline Output Quality](https://qdrant.tech/documentation/search-evaluation/pipeline-output-quality/). It builds a small RAG pipeline over the BioASQ mini benchmark, measures retrieval with recall@10, and scores the generated answers with Ragas. It then changes one thing, the generation prompt, to show how pairing a retrieval score with an answer score tells you which half of the pipeline moved.

You need one [OpenRouter](https://openrouter.ai/) API key. In Colab, add it under **Secrets** as `OPENROUTER_API_KEY`. Retrieval embeddings run locally with FastEmbed, and Qdrant runs in local mode, so there is nothing else to set up.

In [ ]:
%pip install -q "ragas==0.4.3" "langchain-community==0.4.1" "qdrant-client[fastembed]==1.19.1" "openai==3.3.0" datasets pandas

`langchain-community` is pinned because version 0.4.2 removed a module that Ragas 0.4.3 imports, and `import ragas` fails with it.

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata

    os.environ.setdefault("OPENROUTER_API_KEY", userdata.get("OPENROUTER_API_KEY"))
except Exception:
    pass  # not in Colab, or the secret is not set
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

GENERATOR_MODEL = "anthropic/claude-sonnet-5"
JUDGE_MODEL = "openai/gpt-5.4-mini"  # a different model family from the generator
JUDGE_EMBEDDING_MODEL = "openai/text-embedding-3-small"
RETRIEVAL_MODEL = "BAAI/bge-small-en-v1.5"

N = 100  # questions to evaluate
SEED = 42
K = 10  # passages retrieved per question
MAX_CONCURRENCY = 8

RECALL_TARGET = 0.5
FAITHFULNESS_TARGET = 0.8

## Load the benchmark

[rag-mini-bioasq](https://huggingface.co/datasets/rag-datasets/rag-mini-bioasq) ships the three things an end-to-end evaluation needs: a passage corpus, questions labeled with their relevant passage IDs, and a reference answer for each question. The labels drive recall@10, and the reference answers drive `context_precision`.

About 30% of the corpus rows hold the literal string `nan` instead of text. The next cell drops them, drops the labels that point at them, and drops questions left with no label.

In [ ]:
import json
import random

from datasets import load_dataset

corpus = load_dataset("rag-datasets/rag-mini-bioasq", "text-corpus", split="passages")
qa = load_dataset("rag-datasets/rag-mini-bioasq", "question-answer-passages", split="test")

passages = {row["id"]: row["passage"] for row in corpus if row["passage"].strip() != "nan"}

questions = []
dropped_labels = 0
for row in qa:
    labels = json.loads(row["relevant_passage_ids"])
    usable = [pid for pid in labels if pid in passages]
    dropped_labels += len(labels) - len(usable)
    if usable:
        questions.append(
            {"id": row["id"], "question": row["question"], "reference": row["answer"], "relevant": set(usable)}
        )

print(f"{len(passages):,} passages kept, {len(corpus) - len(passages):,} empty rows dropped")
print(f"{len(questions):,} questions kept, {len(qa) - len(questions):,} dropped, {dropped_labels:,} labels pointed at empty rows")

random.seed(SEED)
sample = random.sample(questions, N)

## Index the corpus in Qdrant

Qdrant's built-in FastEmbed integration embeds each `models.Document` locally during upload, so there is no separate embedding step. Expect a few minutes on a CPU runtime.

In [ ]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")
client.create_collection(
    "bioasq",
    vectors_config=models.VectorParams(
        size=client.get_embedding_size(RETRIEVAL_MODEL), distance=models.Distance.COSINE
    ),
)
client.upload_collection(
    "bioasq",
    ids=list(passages),
    vectors=[models.Document(text=text, model=RETRIEVAL_MODEL) for text in passages.values()],
    payload=[{"text": text} for text in passages.values()],
)
print(client.count("bioasq").count, "points")

## Measure retrieval

recall@10 is the share of a question's relevant passages that appear in its top 10 results. It is the retrieval half of the diagnosis.

In [ ]:
def recall_at_k(retrieved_ids, relevant_ids):
    return len(set(retrieved_ids) & relevant_ids) / len(relevant_ids)


assert recall_at_k([1, 2, 3], {1, 9}) == 0.5

for q in sample:
    points = client.query_points(
        "bioasq",
        query=models.Document(text=q["question"], model=RETRIEVAL_MODEL),
        limit=K,
        with_payload=True,
    ).points
    q["retrieved_contexts"] = [p.payload["text"] for p in points]
    q["recall"] = recall_at_k([p.id for p in points], q["relevant"])

print(f"mean recall@{K}: {sum(q['recall'] for q in sample) / N:.3f}")